# Debug List Scraper Radar Malang
List-only. Tidak scrape artikel detail.

In [8]:
from pathlib import Path
import sys
import importlib
import inspect

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'scrapers').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
pd.set_option('display.max_colwidth', 180)
pd.set_option('display.max_rows', 200)

from scrapers.common import cutoff_date, fetch_html, fetch_text, parse_date, normalize_date, is_older_than_cutoff, records_to_df

cutoff = cutoff_date()
print('project:', PROJECT_ROOT)
print('cutoff:', cutoff.date())

import scrapers.radarmalang as scraper
scraper = importlib.reload(scraper)
print('module file:', scraper.__file__)


project: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project
cutoff: 2026-02-28
module file: /Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/scrapers/radarmalang.py


In [9]:
MAX_PAGES = 200
MAX_CLICKS = 40
TITLE_LIMIT = 90


def short_title(value, limit=TITLE_LIMIT):
    text = '' if pd.isna(value) else str(value).strip()
    return text if len(text) <= limit else text[: limit - 3] + '...'


def ensure_debug_columns(df):
    df = df.copy()
    if 'page_num' not in df.columns:
        df['page_num'] = pd.NA
    if 'published_date' not in df.columns:
        df['published_date'] = None
    df['published_dt'] = df['published_date'].apply(parse_date)
    return df


def print_debug_rows(df):
    for _, row in df.iterrows():
        page = row.get('page_num')
        page_text = '---' if pd.isna(page) else f'{int(page):03d}'
        date_text = row.get('published_date')
        date_text = 'None' if pd.isna(date_text) else str(date_text)
        print(f"page={page_text} | date={date_text} | title={short_title(row.get('title'))}")


def audit_list(df):
    df = ensure_debug_columns(df)
    print('total rows:', len(df))
    if len(df) == 0:
        print('No rows found.')
        return df

    print('first page:', df['page_num'].dropna().min() if df['page_num'].notna().any() else None)
    print('last page:', df['page_num'].dropna().max() if df['page_num'].notna().any() else None)
    print('newest date:', df['published_dt'].max())
    print('oldest date:', df['published_dt'].min())
    print('cutoff date:', cutoff)
    print('reached cutoff:', bool((df['published_dt'].dropna() < cutoff).any()))
    print('null parsed dates:', int(df['published_dt'].isna().sum()))

    print('\nrows per month:')
    display(
        df.assign(month=df['published_dt'].dt.to_period('M').astype(str))
        .groupby('month', dropna=False)
        .size()
        .reset_index(name='count')
    )

    print('\nrows per page:')
    display(
        df.groupby('page_num', dropna=False)
        .agg(
            rows=('url', 'count'),
            newest=('published_dt', 'max'),
            oldest=('published_dt', 'min'),
        )
        .reset_index()
        .tail(60)
    )
    return df


In [10]:
def debug_scrape_list(cutoff, max_pages=MAX_PAGES):
    rows = []
    for page_num in range(1, max_pages + 1):
        url = scraper.page_url(page_num)
        print(f'Scraping Radar Malang page {page_num}')
        try:
            html_text = fetch_text(url)
        except Exception as error:
            print(f'Gagal buka Radar Malang page {page_num}: {error}')
            break

        page_data = scraper.extract_data_page(html_text)
        if not page_data:
            print('Radar Malang data-page not found')
            break

        items = page_data['props']['category']['articles']['data']
        if not items:
            break

        stop = False
        for article in items:
            published_date = article.get('date')
            if is_older_than_cutoff(published_date, cutoff):
                stop = True
                break
            rows.append({
                'page_num': page_num,
                'list_page_url': url,
                'title': article.get('title'),
                'url': f"{scraper.BASE_URL}/{article['article_id']}/{article['slug']}",
                'published_date': published_date,
            })
        if stop:
            break
    return records_to_df(rows)

list_df = debug_scrape_list(cutoff)
list_df = ensure_debug_columns(list_df)
print_debug_rows(list_df)


Scraping Radar Malang page 1
Scraping Radar Malang page 2
Scraping Radar Malang page 3
Scraping Radar Malang page 4
Scraping Radar Malang page 5
Scraping Radar Malang page 6
Scraping Radar Malang page 7
Scraping Radar Malang page 8
Scraping Radar Malang page 9
Scraping Radar Malang page 10
Scraping Radar Malang page 11
Scraping Radar Malang page 12
Scraping Radar Malang page 13
Scraping Radar Malang page 14
Scraping Radar Malang page 15
Scraping Radar Malang page 16
Scraping Radar Malang page 17
Scraping Radar Malang page 18
Scraping Radar Malang page 19
Scraping Radar Malang page 20
Scraping Radar Malang page 21
Scraping Radar Malang page 22
Scraping Radar Malang page 23
Scraping Radar Malang page 24
Scraping Radar Malang page 25
Scraping Radar Malang page 26
Scraping Radar Malang page 27
Scraping Radar Malang page 28
Scraping Radar Malang page 29
Scraping Radar Malang page 30
Scraping Radar Malang page 31
Scraping Radar Malang page 32
Scraping Radar Malang page 33
Scraping Radar Mala

In [11]:
list_df = audit_list(list_df)


total rows: 528
first page: 1
last page: 53
newest date: 2026-06-29 04:00:16
oldest date: 2026-02-28 10:20:00
cutoff date: 2026-02-28 08:56:09.113831
reached cutoff: False
null parsed dates: 0

rows per month:


,month,count
0,2026-02,7
1,2026-03,165
2,2026-04,134
3,2026-05,127
4,2026-06,95



rows per page:


,page_num,rows,newest,oldest
0,1,10,2026-06-29 04:00:16,2026-06-26 06:57:14
1,2,10,2026-06-26 04:00:16,2026-06-21 14:52:09
2,3,10,2026-06-21 14:20:04,2026-06-19 04:00:09
3,4,10,2026-06-18 16:38:11,2026-06-16 14:20:43
4,5,10,2026-06-16 04:00:07,2026-06-13 14:15:57
5,6,10,2026-06-13 05:28:24,2026-06-10 16:29:45
6,7,10,2026-06-10 16:21:56,2026-06-08 04:00:02
7,8,10,2026-06-07 12:24:16,2026-06-04 17:21:47
8,9,10,2026-06-04 15:16:06,2026-06-02 17:32:41
9,10,10,2026-06-02 16:29:17,2026-05-31 04:00:20


In [12]:
output_path = PROJECT_ROOT / 'csv' / 'radarmalang_list_debug.csv'
list_df.to_csv(output_path, index=False)
output_path


PosixPath('/Users/tsar/Library/Mobile Documents/com~apple~CloudDocs/main/college/pkl/project/csv/radarmalang_list_debug.csv')

## Optional article sample check

Ambil beberapa artikel detail, cek `content`, lalu simpan ke article debug CSV.

In [ ]:

ARTICLE_SAMPLE_SIZE = 5
ARTICLE_DEBUG_OUTPUT_PATH = PROJECT_ROOT / 'csv' / 'radar_malang_article_debug.csv'

article_rows = []
sample_rows = list_df.head(ARTICLE_SAMPLE_SIZE)
for index, row in sample_rows.iterrows():
    try:
        article = scraper.extract_article(row)
        article_rows.append(article)
        print(f"[{len(article_rows)}] content_len={len(article.get('content') or '')} | {short_title(article.get('title'))}")
    except Exception as error:
        print(f"failed sample article {index + 1}: {row.get('url')} | {error}")

article_df = pd.DataFrame(article_rows)
ARTICLE_DEBUG_OUTPUT_PATH.parent.mkdir(exist_ok=True)
article_df.to_csv(ARTICLE_DEBUG_OUTPUT_PATH, index=False)
print('saved:', ARTICLE_DEBUG_OUTPUT_PATH)
display(article_df[['title', 'published_date', 'content', 'url']].head() if len(article_df) else article_df)
